# Feature Selection Pipeline

This notebook reviews feature-selection as a progression-aligned, stability-aware model-selection component for the FRDA MRI biomarker pipeline.

**Methods reviewed**

| Method | Role | Objective |
|---|---|---|
| `none` | Full-panel reference | Uses all eligible prespecified MRI features. |
| `mi_visit` | Historical MI sensitivity comparator | Ranks features by mutual information with the visit/progression label. `mi` remains a compatibility alias. |
| `mml` | Generic complexity comparator | Greedy forward selection minimizes the existing MML linear-regression score against the visit-label target; this is not progression MML. |
| `progression_univariate` | Progression-aligned filter | Ranks features by annual paired progression effects for V1->V2 and V2->V3 inside the training fold. |
| `progression_mrmr` | Progression-aware redundancy filter | Greedy forward selection combines annual progression relevance with an absolute-correlation redundancy penalty. |
| `sparse_srm` | Embedded sparse SRM-style selector | Uses interval-balanced annual change statistics and ElasticNet-style shrinkage to define non-zero selected coefficients. |

No FARS/SARA or healthy controls are used to select feature-selection methods.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import modelling_pair_count_table
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.metrics import clinical_change_effect_sizes, paired_deltas_from_long, probability_positive_change, reference_effect_sizes
from src.eval.model_selection import select_hierarchical_candidate
from src.eval.stability import selected_feature_jaccard
from src.features.registry import FEATURE_GROUPS
from src.features.selection import feature_domain_coverage, feature_stability_report, feature_set_jaccard_summary
from src.models.srm_global import srm_global_loocv, srm_global_nested_loocv
from src.reporting.tuning_review import tuning_recommendation, tuning_verification_summary

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"
split_group_col = "subject"
selection_k = 8
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300
RANDOM_SEED = DEFAULT_CONFIG.random_state
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows, {len(imaging_cols)} imaging features")
print({"subject_col": subject_col, "split_group_col": split_group_col, "cv_n_splits": CV_N_SPLITS})


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded trackfa_pairs_drop3poms.csv: 414 visit rows, 146 imaging features
{'subject_col': 'pair_id', 'split_group_col': 'subject', 'cv_n_splits': 5}


## 1. Method / Configuration Comparison

Every row below runs the same SRM Global Linear model. The only intended difference is the feature-selection method or selector hyperparameter. Selection is fitted inside each training fold.


In [ ]:
selection_candidates = [
    {"label": "none", "selection_method": "none", "k": len(imaging_cols), "selection_params": {}},
    {"label": "progression_univariate_k8", "selection_method": "progression_univariate", "k": 8, "selection_params": {}},
    {"label": "progression_univariate_k12", "selection_method": "progression_univariate", "k": 12, "selection_params": {}},
    {"label": "progression_mrmr_k8_lam0.25", "selection_method": "progression_mrmr", "k": 8, "selection_params": {"mrmr_redundancy_lambda": 0.25}},
    {"label": "progression_mrmr_k8_lam0.50", "selection_method": "progression_mrmr", "k": 8, "selection_params": {"mrmr_redundancy_lambda": 0.50}},
    {"label": "sparse_srm_lam0.01_alpha0.5", "selection_method": "sparse_srm", "k": 8, "selection_params": {"sparse_lambda": 0.01, "sparse_alpha": 0.5}},
    {"label": "mml", "selection_method": "mml", "k": 8, "selection_params": {}},
    {"label": "mi_visit", "selection_method": "mi_visit", "k": 8, "selection_params": {}},
]

results = []
selected = {}
interval_results = {}
for cand in selection_candidates:
    res = srm_global_loocv(
        long_df,
        imaging_cols,
        subject_col=subject_col,
        visit_col="visit",
        selection_method=cand["selection_method"],
        k=int(cand["k"]),
        cv_n_splits=CV_N_SPLITS,
        random_seed=RANDOM_SEED,
        split_group_col=split_group_col,
        selection_params=cand.get("selection_params", {}),
        compute_ci=False,
    )
    label = cand["label"]
    selected[label] = res["selected_features_by_fold"]
    annual_intervals = adjacent_pair_interval_effect_summary(
        res["oof_df"],
        pair_col=subject_col,
        visit_col="visit",
        score_col="score",
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    interval_results[label] = annual_intervals
    annual_diag = annual_tuning_diagnostics(annual_intervals)
    pooled_deltas = paired_deltas_from_long(res["oof_df"].rename(columns={"score": "value"}), subject_col, "visit", "value")
    stability = feature_set_jaccard_summary(selected[label])
    median_features = int(np.median([len(x) for x in selected[label]])) if selected[label] else 0
    representative_features = selected[label][0] if selected[label] else []
    domains = feature_domain_coverage(representative_features, FEATURE_GROUPS)
    results.append({
        "label": label,
        "selection_method": cand["selection_method"],
        "k/features": cand["k"],
        "selection_params": cand.get("selection_params", {}),
        "pooled_pair_d_z_reference": res["d_score"],
        **annual_diag,
        "pooled_p_progression_reference": probability_positive_change(pooled_deltas),
        "n_subject_pairs": res["n_subjects"],
        "feature_count": median_features,
        "mean_jaccard": stability["mean_jaccard"],
        "median_jaccard": stability["median_jaccard"],
        "iqr_jaccard": stability["iqr_jaccard"],
        "min_jaccard": stability["min_jaccard"],
        "max_jaccard": stability["max_jaccard"],
        "n_domains": domains["n_domains"],
        "represented_domains": domains["represented_domains"],
    })

selection_results = pd.DataFrame(results).sort_values(
    ["mean_validation_annual_dz", "annual_interval_gap"],
    ascending=[False, True],
).reset_index(drop=True)

print("Feature-selection candidates evaluated:", len(selection_results))
print("Top feature-selection candidates")
display(selection_results[[
    "label", "selection_method", "k/features", "dz_v1_v2", "dz_v2_v3",
    "mean_validation_annual_dz", "annual_interval_gap", "p_progression",
    "feature_count", "mean_jaccard", "n_domains", "represented_domains",
]].head(3))


Feature-selection candidates evaluated: 8
Top feature-selection candidates


,label,selection_method,k/features,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,mean_jaccard,n_domains,represented_domains
0,progression_mrmr_k8_lam0.50,progression_mrmr,8,0.905803,0.577164,0.741484,0.328639,0.788300,8,0.800000,2,"structural, structural_ext"
1,progression_univariate_k8,progression_univariate,8,0.912432,0.436133,0.674282,0.476299,0.763468,8,1.000000,2,"structural, structural_ext"
2,progression_mrmr_k8_lam0.25,progression_mrmr,8,0.852794,0.471286,0.662040,0.381508,0.749158,8,0.728889,2,"structural, structural_ext"


## 2. Near-Optimal / Parsimony Review

The recommended configuration is not chosen by raw maximum alone. The review uses the one-SE style hierarchy: near-optimal annual d_z, lower V1->V2/V2->V3 gap, higher P(delta > 0), fewer features, higher feature-selection stability, then coefficient/sign stability if available.


In [ ]:
review_input = selection_results.rename(columns={"label": "param_selection_label"}).copy()
review_input["param_selection_method"] = review_input["selection_method"]
review_input["se_validation_dz"] = selection_results["mean_validation_annual_dz"].std(ddof=1) / np.sqrt(max(len(selection_results), 1))
review_input["jaccard_stability"] = review_input["mean_jaccard"]
review_input["sign_stability"] = np.nan
review_input["score_ranking_stability"] = np.nan
selector_review = tuning_recommendation(review_input)

print("Raw best feature-selection configuration")
display(pd.DataFrame([selector_review["raw_best"]]))
print("One-SE / near-optimal candidate count:", len(selector_review["near_optimal"]))
print("Simplest near-optimal configuration")
display(selector_review["near_optimal"].sort_values(["feature_count", "annual_interval_gap"], ascending=[True, True]).head(1))
print("Most stable near-optimal configuration")
display(selector_review["near_optimal"].sort_values(["jaccard_stability", "feature_count"], ascending=[False, True]).head(1))
print("Recommended feature-selection configuration")
display(pd.DataFrame([selector_review["recommended"]]))
print(selector_review["summary"])
print("Human-verification summary")
display(tuning_verification_summary(selector_review))


Raw best feature-selection configuration


,param_selection_label,param_selection_method,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,progression_mrmr_k8_lam0.50,progression_mrmr,0.905803,0.577164,0.741484,0.328639,0.7883,8,0.8,NaN,NaN,0.086663,1.0


One-SE / near-optimal candidate count: 4
Simplest near-optimal configuration


,param_selection_label,param_selection_method,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,progression_mrmr_k8_lam0.50,progression_mrmr,0.905803,0.577164,0.741484,0.328639,0.7883,8,0.8,NaN,NaN,0.086663,1.0


Most stable near-optimal configuration


,param_selection_label,param_selection_method,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
1,progression_univariate_k8,progression_univariate,0.912432,0.436133,0.674282,0.476299,0.763468,8,1.0,NaN,NaN,0.086663,2.0


Recommended feature-selection configuration


,param_selection_label,param_selection_method,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank,directional_consistency,score_ranking_stability
3,progression_univariate_k12,progression_univariate,0.752878,0.565389,0.659134,0.18749,0.766835,12,0.770989,-inf,-inf,0.086663,4.0,-inf,-inf


Recommended candidate is within one SE of the raw best (performance difference 0.08235) and is preferred by the implemented hierarchy: smaller annual interval gap first, then higher P(delta>0), fewer features, and available stability diagnostics.
Human-verification summary


,item,value
0,Best raw-performance parameters,{'selection_label': 'progression_mrmr_k8_lam0....
1,Recommended parameters,{'selection_label': 'progression_univariate_k1...
2,Difference in performance,0.08235
3,Reason for recommendation,Recommended candidate is within one SE of the ...
4,Any instability/warning,No automatic warning.


## 3. Nested Inner-CV Selector Choice

This section makes feature-selection method itself part of model selection. Each outer fold tunes over the selector candidate space using grouped inner CV and the annual mean d_z hierarchy, then scores the untouched outer fold once.


In [ ]:
nested_candidate_labels = {
    "none",
    "progression_univariate_k8",
    "progression_univariate_k12",
    "progression_mrmr_k8_lam0.25",
    "progression_mrmr_k8_lam0.50",
}
nested_candidates = []
for cand in selection_candidates:
    if cand["label"] not in nested_candidate_labels:
        continue
    row = {
        "ridge": 0.0,
        "covariance_shrinkage": 0.0,
        "z_clip": None,
        "selection_method": cand["selection_method"],
        "k": int(cand["k"]),
        "selection_label": cand["label"],
    }
    row.update(cand.get("selection_params", {}))
    nested_candidates.append(row)

print("Nested candidate subset", nested_candidate_labels)
nested_res = srm_global_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=nested_candidates,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=2,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=False,
    tuning_metric="annual_mean_dz",
)

nested_intervals = adjacent_pair_interval_effect_summary(
    nested_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
nested_diag = annual_tuning_diagnostics(nested_intervals)
nested_stability = feature_set_jaccard_summary(nested_res["selected_features_by_fold"])
print("Nested selector tuning: outer-fold chosen configurations")
display(nested_res["chosen_params_df"])
print("Nested selector tuning: annual OOF interval performance")
display(nested_intervals)
print("Nested selector tuning diagnostics")
display(pd.DataFrame([{**nested_diag, **nested_stability, "outer_cv_d_score": nested_res["d_score"]}]))
print("Chosen selector counts across outer folds")
display(nested_res["chosen_params_df"].get("selection_label", pd.Series(dtype=object)).value_counts().rename_axis("selection_label").reset_index(name="n_outer_folds"))


Nested candidate subset {'progression_mrmr_k8_lam0.25', 'progression_univariate_k12', 'progression_univariate_k8', 'progression_mrmr_k8_lam0.50', 'none'}
Nested selector tuning: outer-fold chosen configurations


,outer_fold,inner_d_score,inner_tuning_metric,inner_dz_v1_v2,inner_dz_v2_v3,inner_annual_interval_gap,inner_p_progression,n_features,ridge,covariance_shrinkage,z_clip,selection_method,k,selection_label,mrmr_redundancy_lambda
0,1,0.645259,annual_mean_dz,0.720593,0.569925,0.150667,0.726190,12,0.0,0.0,None,progression_univariate,12,progression_univariate_k12,NaN
1,2,0.672647,annual_mean_dz,0.855177,0.490118,0.365058,0.756667,8,0.0,0.0,None,progression_mrmr,8,progression_mrmr_k8_lam0.25,0.25
2,3,0.747260,annual_mean_dz,0.861415,0.633104,0.228310,0.773900,12,0.0,0.0,None,progression_univariate,12,progression_univariate_k12,NaN
3,4,0.785572,annual_mean_dz,0.951079,0.620065,0.331015,0.806034,12,0.0,0.0,None,progression_univariate,12,progression_univariate_k12,NaN
4,5,0.651865,annual_mean_dz,0.728945,0.574785,0.154160,0.752469,12,0.0,0.0,None,progression_univariate,12,progression_univariate_k12,NaN


Nested selector tuning: annual OOF interval performance


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,0.739654,0.971614,0.761263,0.612806,0.952921,0.814815
1,V2->V3,99,0.550279,1.008062,0.545879,0.361195,0.762620,0.717172


Nested selector tuning diagnostics


,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,mean_jaccard,median_jaccard,iqr_jaccard,min_jaccard,max_jaccard,outer_cv_d_score
0,0.761263,0.545879,0.653571,0.215385,0.765993,0.697363,0.690476,0.047619,0.538462,0.846154,0.654749


Chosen selector counts across outer folds


,selection_label,n_outer_folds
0,progression_univariate_k12,4
1,progression_mrmr_k8_lam0.25,1


## 4. Comparison Against Full-Feature Reference

This section asks whether selection improves progression sensitivity, or whether it mainly simplifies the model without meaningful performance loss.


In [ ]:
none_row = selection_results[selection_results["selection_method"].eq("none")].head(1)
if len(none_row):
    ref = none_row.iloc[0]
    comparison = selection_results.copy()
    comparison["delta_mean_annual_dz_vs_none"] = comparison["mean_validation_annual_dz"] - ref["mean_validation_annual_dz"]
    comparison["delta_d12_vs_none"] = comparison["dz_v1_v2"] - ref["dz_v1_v2"]
    comparison["delta_d23_vs_none"] = comparison["dz_v2_v3"] - ref["dz_v2_v3"]
    comparison["features_removed_vs_none"] = ref["feature_count"] - comparison["feature_count"]
    comparison["percent_feature_reduction_vs_none"] = 100.0 * comparison["features_removed_vs_none"] / max(float(ref["feature_count"]), 1.0)
    comparison["stability_difference_vs_none"] = comparison["mean_jaccard"] - ref["mean_jaccard"]
    display(comparison[[
        "label", "selection_method", "delta_mean_annual_dz_vs_none", "delta_d12_vs_none",
        "delta_d23_vs_none", "features_removed_vs_none", "percent_feature_reduction_vs_none",
        "stability_difference_vs_none",
    ]])
else:
    print("No full-feature reference row found.")


,label,selection_method,delta_mean_annual_dz_vs_none,delta_d12_vs_none,delta_d23_vs_none,features_removed_vs_none,percent_feature_reduction_vs_none,stability_difference_vs_none
0,progression_mrmr_k8_lam0.50,progression_mrmr,0.236924,0.313767,0.160082,138,94.520548,-0.200000
1,progression_univariate_k8,progression_univariate,0.169722,0.320395,0.019050,138,94.520548,0.000000
2,progression_mrmr_k8_lam0.25,progression_mrmr,0.157480,0.260757,0.054203,138,94.520548,-0.271111
3,progression_univariate_k12,progression_univariate,0.154574,0.160842,0.148306,134,91.780822,-0.229011
4,none,none,0.000000,0.000000,0.000000,0,0.000000,0.000000
5,mml,mml,-0.130255,-0.158960,-0.101550,49,33.561644,-0.082203
6,mi_visit,mi_visit,-0.349228,-0.334915,-0.363541,138,94.520548,-0.876703
7,sparse_srm_lam0.01_alpha0.5,sparse_srm,-0.388344,-0.515342,-0.261345,138,94.520548,-0.511515


## 5. Feature-Level Selection Frequency

Selection frequency and selected-feature lists are stability descriptors only. They should not be interpreted as causal feature importance.


In [ ]:
stability_tables = {}
for label, folds in selected.items():
    table = feature_stability_report(folds, imaging_cols).copy()
    stability_tables[label] = table

freq_parts = []
for label, table in stability_tables.items():
    freq_parts.append(table[["feature", "selection_frequency"]].rename(columns={"selection_frequency": label}))
feature_frequency = freq_parts[0]
for part in freq_parts[1:]:
    feature_frequency = feature_frequency.merge(part, on="feature", how="outer")
method_cols = [c for c in feature_frequency.columns if c != "feature"]
feature_frequency["max_selection_frequency"] = feature_frequency[method_cols].max(axis=1)
feature_frequency = feature_frequency.sort_values("max_selection_frequency", ascending=False, kind="mergesort")
print("Feature-level selection-frequency table")
display(feature_frequency)

print("Selected features by configuration")
selected_feature_rows = []
for label, folds in selected.items():
    table = stability_tables[label]
    chosen = table[table["selection_frequency"] > 0].copy()
    for _, row in chosen.sort_values("selection_frequency", ascending=False).iterrows():
        selected_feature_rows.append({
            "label": label,
            "feature": row["feature"],
            "selection_frequency": row["selection_frequency"],
            "n_selections": row["n_selections"],
        })
display(pd.DataFrame(selected_feature_rows))


Feature-level selection-frequency table


,feature,none,progression_univariate_k8,progression_univariate_k12,progression_mrmr_k8_lam0.25,progression_mrmr_k8_lam0.50,sparse_srm_lam0.01_alpha0.5,mml,mi_visit,max_selection_frequency
0,AD_ACR,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,AD_ALIC,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,AD_CP,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,AD_CST,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,AD_Cing,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
141,sFA_c3c5,1.0,0.0,0.0,0.0,0.0,0.6,0.0,0.0,1.0
142,sMD_c3c5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.4,1.0
143,sRD_c3c5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.2,1.0
144,x3rd_Ventricle,1.0,0.0,0.4,0.2,0.6,0.0,0.0,0.0,1.0


Selected features by configuration


,label,feature,selection_frequency,n_selections
0,none,AD_ACR,1.0,5
1,none,RD_ILF_IFOF,1.0,5
2,none,MD_sCC,1.0,5
3,none,Medulla,1.0,5
4,none,Midbrain,1.0,5
...,...,...,...,...
329,mi_visit,AD_bCC,0.2,1
330,mi_visit,AD_SCR,0.2,1
331,mi_visit,AD_Fx,0.2,1
332,mi_visit,AD_Cing_h,0.2,1


## 6. Interval Tables And Benchmarks

Intervals remain separately observable. V1->V2 and V2->V3 are annual sensitivity checks; the pooled pair d_z is retained only as a reference.


In [ ]:
best = selection_results.iloc[0]
print("Interval table for best feature-selection candidate")
display(interval_results.get(best["label"], pd.DataFrame()))

def benchmark_table(model_name: str, d_score: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            rows.append(hit.iloc[0].to_dict())
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        rows.append(top_img.iloc[0].to_dict())
    return pd.DataFrame(rows)

display(benchmark_table(f"SRM Global Linear ({best['label']})", best["pooled_pair_d_z_reference"]))


Interval table for best feature-selection candidate


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,0.734728,0.811134,0.905803,0.749090,1.127167,0.87963
1,V2->V3,99,0.482197,0.835459,0.577164,0.410825,0.781904,0.69697


,feature,kind,d,source_delta_col,pair_types,mean_diff,sd_diff,n_pairs
0,SRM Global Linear (progression_mrmr_k8_lam0.50),model,0.739225,NaN,NaN,NaN,NaN,NaN
1,FARS,scale,0.407427,delta_mfars_total,"V1V2,V2V3",2.001610,4.912805,207.0
2,SARA,scale,0.405463,delta_sara_total,"V1V2,V2V3",1.070048,2.639077,207.0
3,cerebellumFS,imaging,-0.667820,NaN,NaN,-1615.648449,2419.287337,207.0
